In [ ]:
from syllabifier import Syllabifier
import library
from model import Model

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import numpy as np
import time

# Prepare the Data

In [ ]:
iroyin_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/iroyinspeech_full2_deduped.csv')
print(len(iroyin_full))
multidiac_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/multidiac_full.csv')
print(len(multidiac_full))
yad_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/yad_full_CLEAN.csv')
print(len(yad_full))

In [ ]:
yad_full.loc[262]

In [ ]:
syl = Syllabifier()

# syllabify
print('---Iroyin Sylls---')
iroyin_ngrams = syl.syllabify_df(iroyin_full)
iroyin_ngrams = pd.DataFrame(iroyin_ngrams)
print(len(iroyin_ngrams))

print('---MultiDiac Sylls---')
multidiac_ngrams = syl.syllabify_df(multidiac_full)
multidiac_ngrams = pd.DataFrame(multidiac_ngrams)
print(len(multidiac_ngrams))

print('---YAD Sylls---')
yad_ngrams = syl.syllabify_df(yad_full)
yad_ngrams = pd.DataFrame(yad_ngrams)
print(len(yad_ngrams))

# combine the dataframes
def _create_row(row, df_source, source):
    id = row[0]
    sentence = df_source.loc[id, 'sentence']
    ngrams = row[1]
    return pd.Series([source, id, sentence, ngrams],
                     index=['Source', 'ID', 'Sentence', 'Syllables'])

print('---Iroyin Sylls---')
iroyin_standard = iroyin_ngrams.apply(lambda row: _create_row(row, iroyin_full, 'IroyinSpeech'), axis=1)
print('---MultiDiac Sylls---')
multidiac_standard = multidiac_ngrams.apply(lambda row: _create_row(row, multidiac_full, 'MultiDiac'), axis=1)
print('---YAD Sylls---')
yad_standard = yad_ngrams.apply(lambda row: _create_row(row, yad_full, 'YAD'), axis=1)

print('---Combine ---')
all_letters = pd.concat([iroyin_standard, multidiac_standard, yad_standard], ignore_index=True)


In [ ]:
from enum import Enum
from string import punctuation
import library

class Syllabifier2:
    ####################
    # class variables
    ####################
    # define the possible types of letters
    class Letters(Enum):
        P = -2  # punctuation
        SP = -1 # space
        UNK = 0 # unknown char
        C = 1   # consonant
        M = 2   # m (could be syllabic or consonant)
        N = 3   # n (could be syllabic, consonant, or nasal vowel)
        V = 4   # vowel

    # initialize an instance
    def __init__(self, print_it=False):
        # set instance variables
        self.print_it = print_it

        # train the model

    ####################
    # syllabification helper
    ####################
    # is the given text an alphabetic character?
    def _ischar(self, text):
        return text[0].isalpha()

    # is the given text a vowel, consonant, punctuation, etc?
    def _chartype(self, text):
        if self._ischar(text): 
            if text[0] in ['a', 'e', 'i', 'o', 'u']: return self.Letters.V
            elif text[0] == 'm': return self.Letters.M
            elif text[0] == 'n': return self.Letters.N
            else: return self.Letters.C
        elif text[0] in set(punctuation): return self.Letters.P
        elif text[0].isspace(): return self.Letters.SP
        return self.Letters.UNK # char type unknown

    # syllabify a list of letters, assuming len(letters) != 0
    # return new letters, syllables
    def _get_next_syll(self, letters, syllables):  
        # get char types  
        # get last char's type
        types = [self._chartype(letters[-1])]
        # get second and third to last chars (if present)
        if len(letters) > 2:
            types.insert(0, self._chartype(letters[-2]))
            types.insert(0, self._chartype(letters[-3]))
        # get second to last char, default third to last to SP
        elif len(letters) > 1: 
            types.insert(0, self._chartype(letters[-2]))
            types.insert(0, self.Letters.SP)
        # set other values to default value SP
        else:
            types.insert(0, self.Letters.SP)
            types.insert(0, self.Letters.SP)
        if self.print_it: print('Types:', types)

        # identify the syllable
        # logic based on Kumolalo 2010
        curr_syll = [] # store current identified syllable
        # handle when last thing isn't a letter
        if (types[-1] == self.Letters.SP) or (types[-1] == self.Letters.P):
            curr_syll = ['SP', letters[-1]]
            letters = letters[:-1]
            if self.print_it: print('SP curr_syll : ', curr_syll)
        elif (types[-1] == self.Letters.UNK):
            curr_syll = ['UNK', letters[-1]]
            letters = letters[:-1]
            if self.print_it: print('UNK curr_syll : ', curr_syll)

        # look for CVn (same as DVn in this set up)
        elif (types[-3] in [self.Letters.C, self.Letters.M, self.Letters.N]) and (types[-2] == self.Letters.V) and (types[-1] == self.Letters.N):
            curr_syll = letters[-3:]
            letters = letters[:-3]
            if self.print_it: print('CVn curr_syll : ', curr_syll)
        # look for CV (same as DV)
        elif (types[-2] in [self.Letters.C, self.Letters.M, self.Letters.N]) and (types[-1] == self.Letters.V):
            curr_syll = letters[-2:]
            letters = letters[:-2]
            if self.print_it: print('CV curr_syll : ', curr_syll)
        # look for Vn
        elif (types[-2] == self.Letters.V) and (types[-1] == self.Letters.N):
            curr_syll = letters[-2:]
            letters = letters[:-2]
            if self.print_it: print('Vn curr_syll : ', curr_syll)
        # look for V
        elif types[-1] == self.Letters.V:
            curr_syll = letters[-1:]
            letters = letters[:-1]
            if self.print_it: print('V curr_syll : ', curr_syll)
        # look for N
        elif types[-1] in [self.Letters.N, self.Letters.M]:
            curr_syll = letters[-1:]
            letters = letters[:-1]
            if self.print_it: print('N curr_syll : ', curr_syll)
        # handle other scenario (must be an unknown syllable type)
        else:
            curr_syll = ['UNK', letters[-1]]
            letters = letters[:-1]
            if self.print_it: print('UNK : ', letters[-3:])

        syllables.insert(0,curr_syll) # add syllable to front of list
        return letters, syllables

    # turn letters into syllables
    # combine non-standard syllable types (SP a SP b --> SP a b)
    # return syllables
    def _syllabify_letters(self, letters):
        syllables = []
        # get each syllable
        while (len(letters) > 0):
            if self.print_it: print('_get_next_syll')
            letters, syllables = self._get_next_syll(letters, syllables)

        print(syllables)

        # merge non-standard syllables
        # prev starts out empty, then append label and all letters
        # when it ends, add non-empty one to list of syllables, and reset to empty
        merged_syllables = []
        prev_sp = []
        prev_p = []
        prev_err = []
        prev_unk = []
        for syllable in syllables:
            def _reset(prev_sp, prev_p, prev_err, prev_unk):
                # find non-empty merged non-standard syllable
                if len(prev_sp) > 0: 
                    merged_syllables.append(prev_sp)
                    prev_sp = []
                if len(prev_p) > 0:
                    merged_syllables.append(prev_p)
                    prev_p = []
                if len(prev_err) > 0:
                    merged_syllables.append(prev_err)
                    prev_err = []
                if len(prev_unk) > 0:
                    merged_syllables.append(prev_unk)
                    prev_unk = []
                return prev_sp, prev_p, prev_err, prev_unk

            # collect non-standard syllables but don't add until merged
            if syllable[0] == 'SP':
                if len(prev_sp) == 0: 
                    prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk) # reset possible previous merger (only necessary for first item in SP)
                    prev_sp.append('SP') # add label
                prev_sp.append(syllable[1]) # add subsequent items
            elif syllable[0] == 'P':
                if len(prev_p) == 0: 
                    prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                    prev_p.append('P') # add label
                prev_p.append(syllable[1]) # add subsequent items
            elif syllable[0] == 'ERR':
                if len(prev_err) == 0: 
                    prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                    prev_err.append('ERR') # add label
                prev_err.append(syllable[1]) # add subsequent items
            elif syllable[0] == 'UNK':
                if len(prev_unk) == 0: 
                    prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                    prev_unk.append('UNK') # add label
                prev_unk.append(syllable[1]) # add subsequent items

            # add merged non-standard syllables, current syllable, and reset
            else:
                prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk) # add any merged non-standard syllables
                # add current syllable
                merged_syllables.append(syllable)
        _reset(prev_sp, prev_p, prev_err, prev_unk)        
        return merged_syllables

    ####################
    # syllabification
    ####################
    # syllabify an entire df of text
    # return the syllables
    def syllabify_df(self, df):
        syllables = []
        for id, row in df.iterrows():
            letters = library.get_letters(row['sentence'])
            curr_sylls = self._syllabify_letters(letters)
            syllables.append([id, curr_sylls])
        return syllables        





In [ ]:
syl2 = Syllabifier()
all_letters[all_letters['ID'] == 'yo_f_0108.wav']
test_sentence = all_letters.loc[107, 'Sentence']
print(test_sentence)
letters = library.get_letters(test_sentence)
print(letters)
result = syl2._syllabify_letters(letters)
print(result)

In [ ]:
# Remove duplicates
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped
deduped = dup_rows(all_letters)
display(deduped)

# Test the Tests

In [ ]:
from enum import Enum
import library
import pandas as pd

# train and use a model
class Model2:
    # initialize instance
    def __init__(self, add_diacrtic, eval_diacritic, keep, n, print_it=False):
        self.add_diacritic = add_diacrtic
        self.eval_diacritic = eval_diacritic
        self.keep = keep # which diacritics to maintain (can be 'underdiacs', 'tone', or 'none')
        self.n = n
        self.print_it = print_it
        self.counts = [] # empty at first

    ################
    # HELPERS
    ################ 
    # remove the diacritics from a syllable
    # return the syllable with diacritics removed
    def _rm_diacritics_syll(self, syllable):
        new_syll = []
        # loop through each letter in syllable
        for letter in syllable:
            # corner cases
            if (letter == 'SP') or (letter == 'P'): return syllable
            if (letter == 'ERR') or (letter == 'UNK'): 
                new_syll.append(letter)
                continue
            if (letter[0:2] == 'gb'):
                new_syll.append(letter[0:2])
                continue

            # normal syllable
            include = [letter[0]] # keep original letter
                    
            # check second char
            if (len(letter) > 1) and ((letter[1] in library.UNDERDIACS) and (self.keep=='underdiacs')):
                include.append(letter[1])
            if (len(letter) > 1) and ((letter[1] in library.TONECHARS) and (self.keep=='tone')):
                include.append(letter[1])

            # check third char
            if (len(letter) > 2) and ((letter[2] in library.UNDERDIACS) and (self.keep=='underdiacs')):
                include.append(letter[2])
            if (len(letter) > 2) and ((letter[2] in library.TONECHARS) and (self.keep=='tone')):
                include.append(letter[2])

            # create string from included characters
            new_syll.append(''.join(include))
        return new_syll

    # remove diacritics from a set of syllables
    def rm_diacritics_row(self, row):
        return [self._rm_diacritics_syll(x) for x in row['Syllables']]

    # remove diacritics from df
    # can keep 'underdiacs', 'tone', or 'none'
    def rm_diacritics_df(self, df):
        new_df = df.copy()
        new_df['Syllables'] = new_df.apply(lambda row: self.rm_diacritics_row(row), axis=1)
        return new_df

    # find the n-sized context string for syllable i in syllables
    def _get_context(self, syllables, n, i):
        context = []
        for j in range(1, n+1):
            # get -Syl
            # insert at FRONT of list
            if (i-j >= 0): 
                curr_context = syllables[i-j]
                curr_context = self._rm_diacritics_syll(curr_context)
                if (curr_context[0] == 'SP') or (curr_context[0] == 'ERR') or (curr_context[0] == 'UNK'):
                    curr_context_str = curr_context[0]
                else: curr_context_str = ''.join(curr_context)
                context.insert(0, curr_context_str)
            else: context.insert(0, '<') # start of sentence token

            # get +Syl
            # add to END of list
            if (i+j < len(syllables)):
                curr_context = syllables[i+j]
                curr_context = self._rm_diacritics_syll(curr_context)
                if (curr_context[0] == 'SP') or (curr_context[0] == 'ERR') or (curr_context[0] == 'UNK'):
                    curr_context_str = curr_context[0]
                else: curr_context_str = ''.join(curr_context)
                context.append(curr_context_str)
            else: context.append('>') # end of sentence token
        
        # merge context into a string
        context_str = '.'.join(context) # -Syl.-Syl.+Syl.+Syl
        return context_str

    ################
    # TONE FUNCTIONS
    ################
    # return the index where the tone carrier is located in syllable
    def _tone_carrier_index(self, syllable):
        # find tone carrier in syllable (either N or M alone, or V)
        if len(syllable) == 1: return 0
        
        # if len > 0, tone carrier MUST be a vowel; at most 1 vowel per syll
        else: 
            if (syllable[0][0] in library.VOWELS): return 0
            else: return 1 # vowel MUST be first or second in syllable

    # identify the tone of a syllable
    # returns the tone type
    def _get_tone(self, syllable):
        # corner cases
        if syllable[0] == 'SP' or syllable[0] == 'P' or syllable[0] == 'ERR' or syllable[0] == 'UNK': return 'M'

        tone_carrier = syllable[self._tone_carrier_index(syllable)]

        # get tone from tone carrier
        if (len(tone_carrier) > 1) and (tone_carrier[1] in library.TONECHARS):
            tone = tone_carrier[1]
        elif (len(tone_carrier) > 2) and (tone_carrier[2] in library.TONECHARS):
            tone = tone_carrier[2]
        else: return 'M' # no tone present --> Mid tone

        if tone == library.LOTONE: return 'L'
        elif tone == library.HITONE: return 'H'
        else: return 'M' # default/error is midtone

    # add a given tone to a syllable
    # return syllable with tone added
    def _add_tone(self, syllable, tone):
        if tone == 'H': tone_char = library.HITONE
        elif tone == 'L': tone_char = library.LOTONE
        else: return syllable # mid tone is unmarked

        index = self._tone_carrier_index(syllable)
        new_syll = syllable[:]
        new_syll[index] = ''.join([syllable[index], tone_char])
        return new_syll

    ################
    # DOT FUNCTIONS
    ################

    # determine which letters get dots
    # type = both, vowels, or cons
    def _dots_present(self, syllable):
        dots = []
        for index, letter in enumerate(syllable):
            if (letter[0] in library.DOTCONS):
                # print('DOTCONS', letter, len(letter))
                if (len(letter) > 1) and (letter[1] in library.UNDERDIACS): dots.append('1')
                elif (len(letter) > 2) and (letter[2] in library.UNDERDIACS): dots.append('1')
                else: dots.append('0')
            if (letter[0] in library.DOTVOWELS):
                # print('DOTVOWELS', letter, len(letter))
                if (len(letter) > 1) and (letter[1] in library.UNDERDIACS): dots.append('1')
                elif (len(letter) > 2) and (letter[2] in library.UNDERDIACS): dots.append('1')
                else: dots.append('0')
            if len(dots) <= index: dots.append('0')

        return ' '.join(dots)

    # add given dots to a syllable
    def _add_dots(self, syllable, dots):
        new_syll = []
        dots = dots.split(' ')
        # iterate through letters and add dots
        for index, letter in enumerate(syllable):
            if index >= len(dots): curr_dot = '0'
            else: curr_dot = dots[index]
            if (curr_dot == '1'): 
                curr_char = ''.join([letter, library.UNDERDOT])
            else: curr_char = letter
            new_syll.append(curr_char)
        return new_syll

    ################
    # TRAINING
    ################
    # get n-grams
    # n is the number of items before and after to consider
    def _syll_grams(self, syllables, counts):
        # initialize empty dictionaries
        if not counts: 
            counts = []
            for i in range(self.n+1): counts.append(dict())
        
        # counts has format [n=0, n=1, ..., {syll: {-Syl.-Syl.+Syl.+Syl : {value : count, value : count, value : count}}}]
        # value is Tone:Dots (tone is H M L, dots are 010)
        for i, syll in enumerate(syllables):
            syll_str = ''.join(self._rm_diacritics_syll(syll))
            # corner case
            if (syll[0] == 'SP') or (syll[0] == 'ERR') or (syll[0] == 'UNK'):
                syll_str = syll[0]

            # find all contexts in range 0-n (inclusive)
            for j in range(self.n+1):
                # get contexts for this syllable so far
                poss_contexts = counts[j].get(syll_str, dict())

                # get current context for syllable
                context_str = self._get_context(syllables, j, i)
                context_counts = poss_contexts.get(context_str, dict())

                # find current tone if necessary
                # context_tones = poss_contexts.get(context_str, dict())
                if (self.add_diacritic == 'tone') or (self.add_diacritic == 'both'):
                    curr_tone = self._get_tone(syll)
                else: curr_tone = ' '

                # find current dots if necessary
                if (self.add_diacritic == 'dots') or (self.add_diacritic == 'both'):
                    curr_dot = self._dots_present(syll)
                else: curr_dot = ' '

                # add to counts
                curr_diacs = ':'.join([curr_tone, curr_dot])
                curr_diac_count = context_counts.get(curr_diacs, 0)

                # update all dictionaries
                context_counts.update({curr_diacs : curr_diac_count + 1})
                poss_contexts.update({context_str : context_counts})
                counts[j].update({syll_str : poss_contexts})
        self.counts = counts
        return counts

    # create a full n-gram count from a df
    def create_syll_grams(self, df):
        counts = dict()
        for _, row in df.iterrows():
            counts = self._syll_grams(row['Syllables'], counts)
        self.counts = counts
        return counts

    ################
    # PREDICTIONS
    ################
    # predict the diacritics for each syllable in list of syllables
    def _pred_diacs(self, syllables):
        with_diacs = []
        
        for i, syll in enumerate(syllables):
            # corner case
            if syll[0] == 'SP' or syll[0] == 'ERR':
                with_diacs.append(syll)
                continue

            # get current syllable and its context WITH BACKOFF
            syll_str = ''.join(self._rm_diacritics_syll(syll))
            # try n, n-1, ..., 0 (and stop as soon as it is possible)
            for j in range(self.n, -1, -1):
                # get context
                context_str = self._get_context(syllables, j, i)

                # collect stored counts
                poss_diacs = self.counts[j].get(syll_str, dict()).get(context_str, dict())

                # get most frequent diacs
                pred_tone = ''
                pred_dots = ''
                max_use = -1
                for key,value in poss_diacs.items():
                    # print(key,value)
                    if value > max_use:
                        pred_tone,pred_dots = key.split(':')
                        max_use = value
                    # if values are equal, arbitrarily keep whichever has the dot pattern seen first
                    # if only the tones differ, revert to a default M
                    if value == max_use:
                        n_tone, n_dots = key.split(':')
                        if (n_dots == pred_dots) and (n_tone != pred_tone):
                            pred_tone = 'M'

                # add dots if necessary
                syll_with_diacs = syll
                if (self.add_diacritic == 'dots') or (self.add_diacritic == 'both'):
                    syll_with_diacs = self._add_dots(syll_with_diacs, pred_dots)

                # add tone if necessary
                if (self.add_diacritic == 'tone') or (self.add_diacritic == 'both'):
                    syll_with_diacs = self._add_tone(syll_with_diacs, pred_tone)

                # break if this syllable/sequence of syllables exists in these n-grams
                if not poss_diacs: continue # dictionary is empty, so keep trying
                else: 
                    if(self.print_it) : print('n-gram found, stopping; j = ',j,  syll_str, context_str)
                    break # dictionary was found, so stop going to smaller syllables
            with_diacs.append(syll_with_diacs)

        return with_diacs

    # predict the tone for each syllable in list of syllables
    def _pred_tone(self, syllables):
        with_tones = []
        
        for i, syll in enumerate(syllables):
            # corner case
            if syll[0] == 'SP' or syll[0] == 'ERR':
                with_tones.append(syll)
                continue

            # get current syllable and its context WITH BACKOFF
            syll_str = ''.join(syll)
            new_syll = ''
            # try n, n-1, ..., 0 (and stop as soon as it is possible)
            for j in range(self.n, -1, -1):
                # get context
                context_str = self._get_context(syllables, j, i)

                # collect stored counts
                poss_tones = self.counts[j].get(syll_str, dict()).get(context_str, dict())

                # get most frequent tone
                h = poss_tones.get('H', 0)
                m = poss_tones.get('M', 0)
                l = poss_tones.get('L', 0)

                # add the tone
                if (h > m) and (h > l): new_syll = self._add_tone(syll, 'H')
                elif (l > m) and (l > h): new_syll = self._add_tone(syll, 'L')
                else: new_syll = self._add_tone(syll, 'M')

                # break if this syllable/sequence of syllables exists in these n-grams
                if not poss_tones: continue # dictionary is empty, so keep trying
                else: 
                    if(self.print_it) : print('n-gram found, stopping; j = ',j,  syll_str, context_str)
                    break # dictionary was found, so stop going to smaller syllables
            with_tones.append(new_syll)

        return with_tones
    
    ################
    # MERGING RESULTS
    ################
    # merge diacritics from two columns
    def _merge_diacs_row(self, row, column_1, column_2):
        syllables1 = row[column_1]
        syllables2 = row[column_2]
        merged_syllables = [] # store new merged letters
        # print('syllables: ', syllables1, syllables2)

        # loop through all letters
        for i, syll1 in enumerate(syllables1):
            syll2 = syllables2[i] # get corresponding syllable in second string
            # print('Syllables:', syll1, syll2)
            curr_merged_syll = []

            for j, letter1 in enumerate(syll1):
                letter2 = syll2[j]
                
                orig_letter = letter1[0] # find initial letter
                diacs = set() # store all diacs found

                # find diacs in first column
                letter_len1 = len(letter1)
                # print('len1: ', letter_len1)
                for k in range(1, letter_len1): diacs.add(letter1[k])
                # find diacs in second column
                letter_len2 = len(letter2)
                # print('len2: ', letter_len2)
                for k in range(1, letter_len2): diacs.add(letter2[k])

                # create final letter
                if (diacs): letter_list = [orig_letter] + list(diacs)
                else: letter_list = [orig_letter]
                # print('letter_list: ', letter_list)
                new_letter = ''.join(letter_list)
                # print('new_letter: ', new_letter)
                curr_merged_syll.append(new_letter)
            merged_syllables.append(curr_merged_syll)
        # print(merged_syllables)
        return merged_syllables
    
    def merge_diacs_df(self, df, column_1, column_2):
        new_df = df.copy()
        new_df['Prediction'] = new_df.apply(lambda row: self._merge_diacs_row(row, column_1, column_2), axis=1)
        return new_df

    ################
    # EVALUATION
    ################
    # calculate word error rate for a row, returns (wrong words, total words)
    def _eval_row(self, row):
        correct = row['Syllables']
        pred = row['Prediction']

        wrong_words = 0
        total_words = 0
        in_word = False # identifies whether currently in a word or not
        curr_word_accurate = True # identifies whether the current word has gotten a tone wrong yet

        # iterate through syllables
        for i in range(len(correct)):
            # check if tones match if necessary for eval
            if (self.eval_diacritic == 'tone') or (self.eval_diacritic == 'both'):
                corr_tone = self._get_tone(correct[i])
                pred_tone = self._get_tone(pred[i])
                if corr_tone != pred_tone: 
                    curr_word_accurate = False

            # check if underdots match if necessary for eval
            if (self.eval_diacritic == 'dots') or (self.eval_diacritic == 'both'):
                corr_tone = self._dots_present(correct[i])
                pred_tone = self._dots_present(pred[i])
                if corr_tone != pred_tone: 
                    curr_word_accurate = False

            # check if a word is finished
            if in_word:
                # word has ended
                if correct[i][0] == 'SP':
                    in_word = False
                    if not curr_word_accurate: 
                        wrong_words += 1
                        if self.print_it: print('WRONG', correct,pred)
                    total_words += 1
                    curr_word_accurate = True # reset accuracy
            if correct[i][0] != 'SP': in_word = True
        # corner case: end in a word
        if in_word:
            if not curr_word_accurate: wrong_words+=1
            total_words+=1

        return pd.Series({'Wrong Words' : wrong_words, 'Total Words' : total_words})        

    # determine wrong words in df of syllables
    def evaluate(self, df):
        new_df = df.copy()
        new_df[['Wrong Words', 'Total Words']] = new_df.apply(lambda row: self._eval_row(row), axis=1, result_type='expand')
        return new_df


In [ ]:
# tester = deduped.loc[21583, 'Syllables']
# print(tester)

model = Model2('dots', 'dots', 'tone', 3)

# # for all_syllables[0]:
# #   29 = ['r', 'ọ̀']
# #   2 = ['SP']
# #   6 = ['í']
# #   22 = ['t', 'u', 'n']
# testI = 22
# orig_syll = tester[testI]
# new_syll = model._rm_diacritics_syll(orig_syll)
# print(orig_syll, new_syll)

# print(model._get_tone(orig_syll))
# print(model._get_tone(new_syll))

# minitester = tester[0:50]
# ngram_counts = model._syll_grams(minitester, dict())
# print(ngram_counts)

# untoned = [model._rm_diacritics_syll(x) for x in minitester]
# print(untoned)
# toned = model._pred_diacs(untoned)
# print(toned)

# test merging
test_str = f's{chr(0x0300)}'
print(len(test_str))
for i in range(1, len(test_str)):
    print(i)
    print(test_str[i])
merge_df = pd.DataFrame([[[['a', f's{chr(0x0300)}'], ['i']], [['a', f's{chr(0x0323)}'], ['i']]]], columns=['TonePred', 'DotPred'])
print(merge_df)
merge_df = model.merge_diacs_df(merge_df, 'TonePred', 'DotPred')
print(merge_df)

# Run Final Tests

## Both at Once (Single Model)

In [ ]:
# split data
curr_df = deduped

# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i > 1: break

    print(f"--- Fold {i} ---")
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # set up model
    n = 7
    print_info = 'OneModelBoth'
    tone_model = Model2('both', 'both', 'none', n)

    ################
    # TRAIN
    ################
    # create n grams
    start_time = time.time()
    counts = tone_model.create_syll_grams(train_df)
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)
    # display(counts)

    ################
    # TEST
    ################
    # create detoned version (to test predictions)
    start_time = time.time()
    detoned = test_df.apply(lambda row: tone_model.rm_diacritics_row(row), axis=1)

    # predict tones
    test_df['Prediction'] = detoned.apply(lambda row: tone_model._pred_diacs(row))
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    ################
    # EVALUATE
    ################
    start_time = time.time()
    test_df = tone_model.evaluate(test_df)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    if i == 0: 
        test_df.to_csv(f'{print_info}N{n}I{i}Results.csv')

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

## Dots then Tone

In [ ]:
# split data
curr_df = deduped

# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i > 1: break

    print(f"--- Fold {i} ---")
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # set up model
    n = 7
    print_info = 'DotsThenTone'
    dots_model = Model2('dots', 'both', 'none', n)
    tone_model = Model2('tone', 'both', 'dots', n)

    ################
    # TRAIN
    ################
    # create n grams
    start_time = time.time()
    dot_counts = dots_model.create_syll_grams(train_df)
    tone_counts = tone_model.create_syll_grams(train_df)
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)
    # display(counts)

    ################
    # TEST
    ################
    # create version without diacritics (to test predictions)
    start_time = time.time()
    dediaced = test_df.apply(lambda row: dots_model.rm_diacritics_row(row), axis=1) # dots_model is set to keep 'none'

    # predict dots then tones
    test_df['Prediction1'] = dediaced.apply(lambda row: dots_model._pred_diacs(row))
    test_df['Prediction'] = test_df['Prediction1'].apply(lambda row: tone_model._pred_diacs(row))
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    ################
    # EVALUATE
    ################
    start_time = time.time()
    test_df = tone_model.evaluate(test_df)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    if i == 0: 
        test_df.to_csv(f'{print_info}N{n}I{i}Results.csv')

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

## Tone then Dots

In [ ]:
# split data
curr_df = deduped

# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i == 1: break

    print(f"--- Fold {i} ---")
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # set up model
    n = 7
    print_info = 'ToneThenDots'
    dots_model = Model2('dots', 'both', 'tone', n) # dots happen second so see tone
    tone_model = Model2('tone', 'both', 'none', n) # tone happens first so sees nothing

    ################
    # TRAIN
    ################
    # create n grams
    start_time = time.time()
    dot_counts = dots_model.create_syll_grams(train_df)
    tone_counts = tone_model.create_syll_grams(train_df)
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)
    # display(counts)

    ################
    # TEST
    ################
    # create version without diacritics (to test predictions)
    start_time = time.time()
    dediaced = test_df.apply(lambda row: tone_model.rm_diacritics_row(row), axis=1) # tone_model is set to keep 'none'

    # predict dots then tones
    test_df['Prediction1'] = dediaced.apply(lambda row: tone_model._pred_diacs(row))
    test_df['Prediction'] = test_df['Prediction1'].apply(lambda row: dots_model._pred_diacs(row))
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    ################
    # EVALUATE
    ################
    start_time = time.time()
    test_df = dots_model.evaluate(test_df)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    if i == 0: 
        test_df.to_csv(f'{print_info}N{n}I{i}Results.csv')

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

## Both Seperately

In [ ]:
# split data
curr_df = deduped

# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i == 1: break

    print(f"--- Fold {i} ---")
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # set up model
    n = 7
    print_info = 'BothSeperately'
    dots_model = Model2('dots', 'both', 'none', n)
    tone_model = Model2('tone', 'both', 'none', n)

    ################
    # TRAIN
    ################
    # create n grams
    start_time = time.time()
    dot_counts = dots_model.create_syll_grams(train_df)
    tone_counts = tone_model.create_syll_grams(train_df)
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)
    # display(counts)

    ################
    # TEST
    ################
    # create version without diacritics (to test predictions)
    start_time = time.time()
    dediaced = test_df.apply(lambda row: dots_model.rm_diacritics_row(row), axis=1) # dots_model is set to keep 'none'

    # predict dots and tones
    test_df['DotPred'] = dediaced.apply(lambda row: dots_model._pred_diacs(row))
    test_df['TonePred'] = dediaced.apply(lambda row: tone_model._pred_diacs(row))
    # merge results
    test_df = dots_model.merge_diacs_df(test_df, 'DotPred', 'TonePred')
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    ################
    # EVALUATE
    ################
    start_time = time.time()
    test_df = tone_model.evaluate(test_df)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    if i == 0: 
        test_df.to_csv(f'{print_info}N{n}I{i}Results.csv')

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

# Testing For Each Dataset

In [ ]:
sources = ['IroyinSpeech', 'MultiDiac', 'YAD']

## Both at Once

In [ ]:
for source in sources:
    print(f'----------\n{source}\n----------\n')
    # split data
    curr_df = deduped[deduped['Source'] == source]

    # split into 10 folds, make it even across all three datasets
    skf = StratifiedKFold(n_splits=10,shuffle=False)
    X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
    y = curr_df['Source'].to_numpy() # stratify across the datasets

    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i > 1: break

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # set up model
        n = 2
        print_info = f'OneModelBoth'
        tone_model = Model2('both', 'both', 'none', n)

        ################
        # TRAIN
        ################
        # create n grams
        start_time = time.time()
        counts = tone_model.create_syll_grams(train_df)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)
        # display(counts)

        ################
        # TEST
        ################
        # create detoned version (to test predictions)
        start_time = time.time()
        detoned = test_df.apply(lambda row: tone_model.rm_diacritics_row(row), axis=1)

        # predict tones
        test_df['Prediction'] = detoned.apply(lambda row: tone_model._pred_diacs(row))
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        ################
        # EVALUATE
        ################
        start_time = time.time()
        test_df = tone_model.evaluate(test_df)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)

        if i == 0: 
            test_df.to_csv(f'{print_info}N{n}I{i}Results{source}.csv')

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))

## Dots Then Tone

In [ ]:
for source in sources:
    print(f'----------\n{source}\n----------\n')

    # split data
    curr_df = deduped[deduped['Source'] == source]

    # split into 10 folds, make it even across all three datasets
    skf = StratifiedKFold(n_splits=10,shuffle=False)
    X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
    y = curr_df['Source'].to_numpy() # stratify across the datasets

    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i > 1: break

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # set up model
        n = 4
        print_info = 'DotsThenTone'
        dots_model = Model2('dots', 'both', 'none', n)
        tone_model = Model2('tone', 'both', 'dots', n)

        ################
        # TRAIN
        ################
        # create n grams
        start_time = time.time()
        dot_counts = dots_model.create_syll_grams(train_df)
        tone_counts = tone_model.create_syll_grams(train_df)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)
        # display(counts)

        ################
        # TEST
        ################
        # create version without diacritics (to test predictions)
        start_time = time.time()
        dediaced = test_df.apply(lambda row: dots_model.rm_diacritics_row(row), axis=1) # dots_model is set to keep 'none'

        # predict dots then tones
        test_df['Prediction1'] = dediaced.apply(lambda row: dots_model._pred_diacs(row))
        test_df['Prediction'] = test_df['Prediction1'].apply(lambda row: tone_model._pred_diacs(row))
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        ################
        # EVALUATE
        ################
        start_time = time.time()
        test_df = tone_model.evaluate(test_df)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)

        if i == 0: 
            test_df.to_csv(f'{print_info}N{n}I{i}Results{source}.csv')

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))

## Tone Then Dots

In [ ]:
for source in sources:
    print(f'----------\n{source}\n----------\n')
    
    # split data
    curr_df = deduped[deduped['Source'] == source]

    # split into 10 folds, make it even across all three datasets
    skf = StratifiedKFold(n_splits=10,shuffle=False)
    X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
    y = curr_df['Source'].to_numpy() # stratify across the datasets

    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i == 1: break

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # set up model
        n = 2
        print_info = 'ToneThenDots'
        dots_model = Model2('dots', 'both', 'tone', n) # dots happen second so see tone
        tone_model = Model2('tone', 'both', 'none', n) # tone happens first so sees nothing

        ################
        # TRAIN
        ################
        # create n grams
        start_time = time.time()
        dot_counts = dots_model.create_syll_grams(train_df)
        tone_counts = tone_model.create_syll_grams(train_df)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)
        # display(counts)

        ################
        # TEST
        ################
        # create version without diacritics (to test predictions)
        start_time = time.time()
        dediaced = test_df.apply(lambda row: tone_model.rm_diacritics_row(row), axis=1) # tone_model is set to keep 'none'

        # predict dots then tones
        test_df['Prediction1'] = dediaced.apply(lambda row: tone_model._pred_diacs(row))
        test_df['Prediction'] = test_df['Prediction1'].apply(lambda row: dots_model._pred_diacs(row))
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        ################
        # EVALUATE
        ################
        start_time = time.time()
        test_df = dots_model.evaluate(test_df)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)

        if i == 0: 
            test_df.to_csv(f'{print_info}N{n}I{i}Results{source}.csv')

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))

## Both Seperately

In [ ]:
for source in sources:
    print(f'----------\n{source}\n----------\n')
    
    # split data
    curr_df = deduped[deduped['Source'] == source]

    # split into 10 folds, make it even across all three datasets
    skf = StratifiedKFold(n_splits=10,shuffle=False)
    X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
    y = curr_df['Source'].to_numpy() # stratify across the datasets

    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i == 1: break

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # set up model
        n = 4
        print_info = 'BothSeperately'
        dots_model = Model2('dots', 'both', 'none', n)
        tone_model = Model2('tone', 'both', 'none', n)

        ################
        # TRAIN
        ################
        # create n grams
        start_time = time.time()
        dot_counts = dots_model.create_syll_grams(train_df)
        tone_counts = tone_model.create_syll_grams(train_df)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)
        # display(counts)

        ################
        # TEST
        ################
        # create version without diacritics (to test predictions)
        start_time = time.time()
        dediaced = test_df.apply(lambda row: dots_model.rm_diacritics_row(row), axis=1) # dots_model is set to keep 'none'

        # predict dots and tones
        test_df['DotPred'] = dediaced.apply(lambda row: dots_model._pred_diacs(row))
        test_df['TonePred'] = dediaced.apply(lambda row: tone_model._pred_diacs(row))
        # merge results
        test_df = dots_model.merge_diacs_df(test_df, 'DotPred', 'TonePred')
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        ################
        # EVALUATE
        ################
        start_time = time.time()
        test_df = tone_model.evaluate(test_df)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)

        if i == 0: 
            test_df.to_csv(f'{print_info}N{n}I{i}Results{source}.csv')

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))